## NetCDF to Parquet conversion

DuckDB does not read NetCDF natively. This notebook converts the downloaded SST data into a flat table (Parquet), which DuckDB can query directly and efficiently.

In [1]:
import xarray as xr
import pandas as pd
import os
import duckdb

In [2]:
# Open the full dataset
ds = xr.open_dataset("../data/raw/med_sst_2016_2026.nc")

# Try to convert to table for the first day, to see the resulting shape
sample = ds.isel(time=0).to_dataframe().reset_index()
print(sample.shape)
print(sample.head())

(19120, 4)
    latitude  longitude  analysed_sst       time
0  40.507652   2.043520    288.579994 2016-01-01
1  40.507652   2.093567    288.649994 2016-01-01
2  40.507652   2.143612    288.699994 2016-01-01
3  40.507652   2.193659    288.739994 2016-01-01
4  40.507652   2.243704    288.749994 2016-01-01


In [3]:
# Convert the entire dataset to a flat table (may take a few minutes)
df = ds.to_dataframe().reset_index()

print(df.shape)
print(df.head())

(73134000, 4)
        time   latitude  longitude  analysed_sst
0 2016-01-01  40.507652   2.043520    288.579994
1 2016-01-01  40.507652   2.093567    288.649994
2 2016-01-01  40.507652   2.143612    288.699994
3 2016-01-01  40.507652   2.193659    288.739994
4 2016-01-01  40.507652   2.243704    288.749994


In [4]:
# Remove NaN rows (land cells), we already confirmed they are 37.57% of the grid
df_clean = df.dropna(subset=["analysed_sst"])

print(df_clean.shape)
print(f"Rows removed: {len(df) - len(df_clean)}")
print(f"Percentage removed: {(len(df) - len(df_clean)) / len(df) * 100:.2f}%")

(45655200, 4)
Rows removed: 27478800
Percentage removed: 37.57%


In [5]:
# Save the cleaned table as Parquet, ready for DuckDB
df_clean.to_parquet("../data/processed/med_sst_2016_2026.parquet")

In [6]:
# Check the size of the Parquet file
file_size_mb = os.path.getsize("../data/processed/med_sst_2016_2026.parquet") / (1024 * 1024)
print(f"Parquet file size: {file_size_mb:.2f} MB")

Parquet file size: 249.38 MB


**Note on file size**: the Parquet file (249.38 MB) is larger than the source NetCDF (139.58 MB), despite Parquet's compression. This is because the long-format table repeats latitude and longitude values on every row, while NetCDF stores coordinates once and reconstructs the grid structure implicitly. Pandas also defaults to float64 for all columns, adding further size compared to the float32 used for coordinates in the original file. This tradeoff (larger, flatter, SQL-queryable format vs compact, grid-based format) is expected and acceptable at this scale; it is not optimized further here.

## Loading into DuckDB

Quick verification query to confirm DuckDB reads the Parquet file correctly, before moving to real exploratory analysis in the next notebook.

In [7]:
# Connect to an in-memory DuckDB and query the Parquet file directly
con = duckdb.connect()

result = con.execute("""
    SELECT COUNT(*) AS n_rows,
           MIN(time) AS first_day,
           MAX(time) AS last_day,
           MIN(analysed_sst) AS min_sst,
           MAX(analysed_sst) AS max_sst
    FROM '../data/processed/med_sst_2016_2026.parquet'
""").df()

print(result)

     n_rows  first_day   last_day     min_sst     max_sst
0  45655200 2016-01-01 2026-06-21  277.399994  303.809993


**Results**: 45,655,200 rows, consistent with the 37.57% NaN figure established in the previous notebook (applied here to the row-level table for the first time). Date range correct (2016-01-01 to 2026-06-21). Temperature range 277.40 to 303.81 K (4.25 to 30.66 C). The low end is consistent with cold winters in the Gulf of Lion. The high end is notably above typical Mediterranean summer temperatures (around 24-26 C historically) and close to the record-breaking marine heatwave peaks observed in the region in 2024 and 2025 (up to 30.8 C off the Côte d'Azur), suggesting the dataset already contains at least one real heatwave event, which will be useful for validating the detection logic later.